<a href="https://colab.research.google.com/github/hUSsAin976-tech/ML-internship-at-FlyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Ranking / opportunity scoring — built on top of exploratory signal analysis (EDA). Explicitly NOT classification.**

My lane is the **AI Referral Opportunity** freestyle direction: *"What broad content patterns appear around AI-referred traffic, and where might AI visibility be improved?"*

Why not classification, even though "will this page get AI referral traffic" sounds like a yes/no question: the lane guide is explicit that AI-session data is **sparse** — in the full warehouse it's 30,177 rows with AI sessions against 78.8M daily rows, and I confirm the same shape below in the starter slice. With positives this rare, a binary classifier trained on `has_ai_sessions` would report impressive-looking AUC while mostly memorizing noise from a handful of clients — a textbook case of a metric that looks good and means nothing.

So the honest task type here is:
- **EDA / signal analysis** — first, to see whether any observed signals associate with AI referral at all, and how strongly.
- **Ranking / opportunity scoring** — second, to turn any real association into an ordered review queue: which pages, among those with real search demand but zero observed AI referral, look most like the pages that *do* get AI referral traffic. A reviewer works down this list; it is decision-support, not a forecast.

This keeps the promise the lane guide makes for this direction: *"EDA/ranking only by default; positive examples are sparse."*


In [3]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/hUSsAin976-tech/ML-internship-at-FlyRank/main/data/raw/content_refresh_anonymized.csv")

# Observed AI-referral flag: did this page get ANY AI-referred sessions in the trailing 90 days?
df["has_ai_sessions"] = (df["ai_sessions_90d"] > 0).astype(int)

n_rows = len(df)
base_rate = df["has_ai_sessions"].mean()

print(f"rows in starter dataset: {n_rows:,}")
print(f"rows with ai_sessions_90d > 0: {df['has_ai_sessions'].sum():,} ({base_rate:.1%} of all rows)")
print(f"rows with ai_sessions_90d == 0: {(df['has_ai_sessions']==0).sum():,} ({1-base_rate:.1%} of all rows)")


rows in starter dataset: 30,000
rows with ai_sessions_90d > 0: 1,930 (6.4% of all rows)
rows with ai_sessions_90d == 0: 28,070 (93.6% of all rows)


## 2. Target or proxy

**No single trained target. The proxy signal is `has_ai_sessions` (`ai_sessions_90d > 0`), used only for observation and ranking evidence — never as a label to fit a classifier against.**

- `has_ai_sessions` is an **observed** fact, not something someone defined by rule — it comes straight from GA4 session data (real click-throughs from AI assistants, not citations or rankings, per the data dictionary). That part is clean.
- What makes it unsafe as a *training target* is the base rate: only about 6% of pages show any AI-referred sessions at all (confirmed below). A model trained to predict that label would mostly be learning "did this page happen to be one of the ~6% with any AI traffic," which is dominated by noise at this sample size — the lane guide calls this out by name as a mistake to avoid.
- Instead, `has_ai_sessions` becomes the **evidence variable** I use two ways:
  1. **Association check** — do pages with `has_ai_sessions == 1` differ systematically (by content type, intent, demand, structure) from pages with `has_ai_sessions == 0`? That's EDA, not prediction.
  2. **Ranking validation** — once I build a transparent opportunity score from safe signals, I check whether pages that score high are disproportionately the ones that already show AI referral (a lift check), as evidence the score is picking up something real rather than random.

The actual **deliverable proxy** is an **opportunity score** for the ~91–92% of demand-worthy pages with zero observed AI sessions — ranking them by how similar their content and demand profile looks to the pages that do get AI referral. That score is the thing a reviewer acts on, not a decline/no-decline label.


In [4]:
# Confirm the sparsity that rules out a classifier, and size the "zero-AI opportunity pool"

# Demand-worthy pool: real search demand and at least some GA4 traffic, same spirit as
# the starter pipeline's own filter and the dictionary's `measurable_opportunity` flag.
pool = df[(df["impressions_90d"] >= 100) & (df["sessions_90d"] > 0)]

pool_zero_ai_share = (pool["ai_sessions_90d"] == 0).mean()

print(f"demand-worthy pool (impressions_90d >= 100 and sessions_90d > 0): {len(pool):,} rows "
      f"({len(pool)/n_rows:.1%} of the dataset)")
print(f"of those, share with ZERO observed AI-referred sessions: {pool_zero_ai_share:.1%}")
print()
print("Reading: this pool is exactly the candidate pool for the opportunity queue —")
print("real demand exists, but ~92% show no AI referral yet. Base positive rate (6.4%) is far")
print("too thin for a supervised classifier to be trustworthy, matching the lane guide's warning.")


demand-worthy pool (impressions_90d >= 100 and sessions_90d > 0): 22,006 rows (73.4% of the dataset)
of those, share with ZERO observed AI-referred sessions: 91.8%

Reading: this pool is exactly the candidate pool for the opportunity queue —
real demand exists, but ~92% show no AI referral yet. Base positive rate (6.4%) is far
too thin for a supervised classifier to be trustworthy, matching the lane guide's warning.


## 3. Success metric

**Lift at top-K, not AUC or accuracy.**

Because this is a ranking/opportunity task validated against a sparse observed signal — not a classifier — the right metric asks: *"Does my score concentrate the pages that already show AI referral higher than chance would?"*

```
lift@K = (share of has_ai_sessions==1 pages in the top-K of my ranked score)
         / (base rate of has_ai_sessions==1 across the whole demand pool)
```

- A lift@K near 1.0 means the score is doing no better than a random shuffle — the ranking has no signal.
- A lift@K meaningfully above 1.0 means the top of my queue is genuinely enriched for pages that resemble ones with real AI referral — that's the evidence a plain rule wouldn't reliably produce.

I don't chase precision/recall/ROC-AUC here on purpose: those assume the label is something worth predicting forward in time, and with 6% positives and no time-window separation, those numbers would be unstable and easy to over-read. Lift@K is honest about what this lane can and can't claim — association and ranking, not forecasting.


In [5]:
# Compute the metric I can defend TODAY, on the baseline signal itself, as the guide instructs:
# "Check the metric exists in your data. Can you compute it today, on a baseline? Do it."

# Baseline "rule": rank the demand pool by informational/commercial intent alignment,
# a cheap single-signal proxy I can beat later with a real multi-signal score (ML-07/ML-08).
baseline_rank = pool.sort_values("main_intent", key=lambda s: s.map(
    {"navigational": 0, "commercial": 1, "informational": 2, "transactional": 3}
))

k = 500
top_k = baseline_rank.head(k)
lift_at_k = top_k["has_ai_sessions"].mean() / pool["has_ai_sessions"].mean() if pool["has_ai_sessions"].mean() else float("nan")

print(f"pool base rate of has_ai_sessions: {pool['has_ai_sessions'].mean():.1%}")
print(f"top-{k} (by this single-signal baseline) rate of has_ai_sessions: {top_k['has_ai_sessions'].mean():.1%}")
print(f"lift@{k}: {lift_at_k:.2f}x")
print()
print("This trivial single-signal baseline is close to 1x lift, as expected — one column")
print("shouldn't beat chance by much. It's the honest floor a real multi-signal opportunity")
print("score (built in a later assignment) needs to clear.")


pool base rate of has_ai_sessions: 8.2%
top-500 (by this single-signal baseline) rate of has_ai_sessions: 8.0%
lift@500: 0.98x

This trivial single-signal baseline is close to 1x lift, as expected — one column
shouldn't beat chance by much. It's the honest floor a real multi-signal opportunity
score (built in a later assignment) needs to clear.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page), summarized over its trailing 90-day window.**

Same grain as the rest of the starter dataset (`content_id` + `client_id`), which keeps this lane comparable to the other lanes and lets `client_id` be used for grouped validation later — never as a feature.


In [6]:
cols = [
    "content_id", "client_id", "content_type", "main_intent", "word_count",
    "impressions_90d", "sessions_90d", "ai_sessions_90d", "ai_traffic_pct", "has_ai_sessions",
]

# Show the unit of analysis: a handful of real rows from the zero-AI opportunity pool
# (the ones a reviewer would actually see in the queue) next to a few rows that DO show AI referral,
# so the contrast is visible.
zero_ai_sample = pool[pool["has_ai_sessions"] == 0][cols].sample(3, random_state=7)
has_ai_sample = pool[pool["has_ai_sessions"] == 1][cols].sample(3, random_state=7)

display(pd.concat([has_ai_sample, zero_ai_sample], keys=["has AI referral", "opportunity pool (zero AI referral)"]))


content_id  \
has AI referral                     1178   content_9edb1f8ff902   
                                    17590  content_f742d659d8fa   
                                    26010  content_f7abac02c1f6   
opportunity pool (zero AI referral) 16091  content_5692e949daf8   
                                    27521  content_093f50415d7b   
                                    24734  content_46c9510685ef   

                                                   client_id     content_type  \
has AI referral                     1178   client_6208ef0f77  keyword article   
                                    17590  client_6208ef0f77  keyword article   
                                    26010  client_6208ef0f77  keyword article   
opportunity pool (zero AI referral) 16091  client_3fdba35f04  keyword article   
                                    27521  client_4ec9599fc2  keyword article   
                                    24734  client_7f2253d7e2  keyword article   

                                             main_intent  word_count  \
has AI referral                     1178   informational      6884.0   
                                    17590  informational      6344.0   
                                    26010  informational      6845.0   
opportunity pool (zero AI referral) 16091  informational      1465.0   
                                    27521     commercial         NaN   
                                    24734  informational      2760.0   

                                           impressions_90d  sessions_90d  \
has AI referral                     1178             35073           247   
                                    17590            27349            64   
                                    26010            10221           464   
opportunity pool (zero AI referral) 16091              429             7   
                                    27521             2475            21   
                                    24734             3393             3   

                                           ai_sessions_90d  ai_traffic_pct  \
has AI referral                     1178                 2            0.81   
                                    17590                2            3.13   
                                    26010                2            0.43   
opportunity pool (zero AI referral) 16091                0            0.00   
                                    27521                0            0.00   
                                    24734                0            0.00   

                                           has_ai_sessions  
has AI referral                     1178                 1  
                                    17590                1  
                                    26010                1  
opportunity pool (zero AI referral) 16091                0  
                                    27521                0  
                                    24734                0

## 5. Why ML/data-driven scoring beats a fixed rule here

A single if-statement can't hold this pattern for two separate reasons, one about the *signal* and one about the *label*:

1. **The signal is genuinely multi-factor, not one threshold.** AI referral likelihood plausibly depends on content type, intent, structure (word count, informational vs. transactional framing), and existing demand — all at once, interacting. A rule like "flag if `word_count > 1200`" or "flag if `main_intent == 'informational'`" only nudges the rate a little (checked below) — no single column cleanly separates the ~6% with AI referral from the ~94% without. That's the classic case for combining several weak, correlated signals into one score instead of hand-picking one column and a cutoff.

2. **But the label itself is too sparse for full supervised ML, so the "beats a fixed rule" bar has to be honest about scope.** I can't point to a trained classifier's AUC as proof, because the lane guide is right that this label doesn't support one yet. What a weighted, multi-signal *score* can still do — and a single rule can't — is rank the 92% zero-AI pool in an order that's demonstrably better than chance (lift@K > 1), by blending several observed signals with a transparent, checkable weighting. That's a real step up from "pick one column" without overclaiming a forecast the sparse label can't support.

The honest framing, in the paragraph from `framing-ml-problems`:

> For a content reviewer deciding which pages to check first for AI-visibility structure (FAQ blocks, clear definitions, source clarity), we will build a **ranked opportunity queue** from **observed GSC/GA4 signals in the starter dataset**, scoring pages by how closely their content/demand profile resembles pages that already show AI referral, measured by **lift@K against the observed `has_ai_sessions` base rate**. A wrong call costs a reviewer's limited time on a page that was never a real AI-visibility opportunity — low-stakes, so a directional ranking is an acceptable trade. A plain single-column rule isn't enough because no single signal cleanly separates AI-referred pages from the rest, but a full supervised classifier isn't earned yet either because positives are too sparse — a transparent multi-signal score sits at the right level of complexity for this data, right now. We will claim only observed and directional results, never predictions of future AI citations or rankings.


In [7]:
# Quick check: does any single column separate the classes cleanly? (motivates the multi-signal score)
for col, cond_name, cond in [
    ("word_count", "word_count > 1200", df["word_count"] > 1200),
    ("main_intent", "main_intent == 'informational'", df["main_intent"] == "informational"),
    ("impressions_90d", "impressions_90d >= 500", df["impressions_90d"] >= 500),
]:
    rate_in = df.loc[cond, "has_ai_sessions"].mean()
    rate_out = df.loc[~cond, "has_ai_sessions"].mean()
    print(f"{cond_name:35s} -> has_ai_sessions rate: {rate_in:.1%} (in)  vs  {rate_out:.1%} (out)  |  overall: {base_rate:.1%}")


word_count > 1200                   -> has_ai_sessions rate: 7.9% (in)  vs  3.0% (out)  |  overall: 6.4%
main_intent == 'informational'      -> has_ai_sessions rate: 6.4% (in)  vs  6.5% (out)  |  overall: 6.4%
impressions_90d >= 500              -> has_ai_sessions rate: 10.1% (in)  vs  1.8% (out)  |  overall: 6.4%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.